# Ordered Logistic Regression Predictors Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following the Croissant schema.

### Dataset Source
The dataset is defined by a [Croissant schema](https://github.com/mlcommons/croissant) at this URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and entire Croissant ontology using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset package
dataset = mlc.Dataset(croissant_url)
# Access structured metadata as a Python object
metadata = dataset.metadata

# Show basic dataset information
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Authors: {metadata.author}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Review the available record sets and their fields. Each entity (`RecordSet`, `Field`, `Column`) is referenced by its `@id`, which uniquely identifies it in the Croissant schema.

In [ ]:
# List all available record sets in the dataset, referenced by their @id
record_sets = [r for r in dataset.record_sets]

for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name', '(no name)')}")
    print("  Fields:")
    for field in rs.get('field', []):
        print(f"    - Field @id: {field['@id']}, name: {field.get('name', '(no name)')}")
    print("  Columns:")
    for col in rs.get('column', []):
        print(f"    - Column @id: {col['@id']}, name: {col.get('name', '(no name)')}")
    print("")

## 3. Data Extraction
Load all records from each available record set into a pandas DataFrame. Use each record set's `@id`.

In [ ]:
# Choose the record set(s) to load, referenced by their @id
record_set_ids = [rs['@id'] for rs in record_sets]
# For demonstration, we will show the first available record set
if not record_set_ids:
    raise ValueError("No record sets are defined in the Croissant schema.")

# Load each record set into DataFrames, using @id for keys
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame.from_records(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from record set @id: {rs_id}")

# Show columns present in the first record set as an example
first_rs_id = record_set_ids[0]
print(f"\nColumns in first record set ({first_rs_id}):")
print(dataframes[first_rs_id].columns.tolist())

# Preview the first few records
dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
As an example, filter, normalize, and group numeric data using the `@id` of columns. This approach gives you a template for analyzing each record set.

In [ ]:
# Identify numeric columns for analysis using their @id
import numpy as np

df = dataframes[first_rs_id]
# Find which columns are likely numeric (simple heuristic based on dtype)
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_columns:
    numeric_field_id = numeric_columns[0]  # Take the first numeric column as example
    print(f"Using numeric field (column @id): {numeric_field_id}\n")

    threshold = df[numeric_field_id].mean()  # Use mean as an example threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    print(filtered_df.head())

    norm_col_name = f"{numeric_field_id}_normalized"
    filtered_df[norm_col_name] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} (z-score) for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col_name]].head())

    # Try grouping by a likely categorical column
    possible_group_fields = df.select_dtypes(include=[object]).columns.tolist()
    group_field = None
    for col in possible_group_fields:
        # Choose a group field with less than 10 unique values as a likely candidate
        if df[col].nunique() > 1 and df[col].nunique() < 10:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"\nGrouped data by {group_field} (@id):")
        print(grouped_df)
    else:
        print("No suitable group field found.")
else:
    print("No numeric fields found for EDA in the first record set.")

## 5. Visualization
Plot the distribution of a numeric field and visualize relationships using only column `@id`s.

In [ ]:
import matplotlib.pyplot as plt

if 'numeric_field_id' in locals() and len(filtered_df) > 0:
    plt.figure(figsize=(6,4))
    filtered_df[numeric_field_id].hist(bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(7,4))
        filtered_df.boxplot(column=numeric_field_id, by=group_field)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No visualization can be shown as there are no numeric fields or filtered data available.")

## 6. Conclusion
We have used the `mlcroissant` library to:
- Load metadata and records from the Croissant schema, referring to all entities by their `@id`.
- Explore available record sets and fields.
- Extract records into pandas DataFrames using their Croissant `@id`s.
- Run basic EDA steps such as filtering, normalization, and grouping by a categorical field (all using only field/column `@id`s).
- Visualize data using field `@id`.

You may wish to extend this analysis by mapping field `@id`s to human-readable names based on the Croissant schema, engineering new features, or applying machine learning methods for deeper exploration.